In [ ]:

features_to_drop = [
    "actual_default",
    "ever_defaulted",
    "segment_risk_score",
    "rider_id"
]

X = df.drop(columns=features_to_drop)


In [49]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
import joblib
import os
import warnings
import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, classification_report,
    confusion_matrix, precision_score,
    recall_score, f1_score, roc_curve
)
from xgboost.callback import EarlyStopping
from lightgbm import early_stopping, log_evaluation
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score




In [2]:
!pip install xgboost

In [3]:
!pip install lightgbm

In [4]:
df = pd.read_csv("features.csv")

# Load feature column names
with open("feature_cols.json", "r") as f:
    FEATURE_COLS = json.load(f)

In [4]:
df.head()

,rider_id,avg_daily_income,total_income_90d,income_volatility_cv,income_trend_90d,rain_season_dip,income_gap_max_days,monthly_income_estimate,fuel_spend_monthly,fuel_to_income_ratio,...,app_peak_hour_ratio,app_income_stability_cv,actual_default,requested_amount,requested_term_days,loan_purpose_code,loan_to_income_ratio,loan_to_ndi_ratio,requested_monthly_repayment,repayment_to_ndi_ratio
0,RDR-00001,912.588409,80307.7800,0.179872,0.040904,0.247453,1,26769.260,8728.20,0.3261,...,0.4312,0.3042,0,10000,90,3,0.3736,0.8369,3333.33,0.2790
1,RDR-00002,950.066744,81705.7400,0.198440,0.037983,0.221748,1,27235.250,7151.31,0.2626,...,0.4339,0.3006,0,15000,60,1,0.5508,1.2136,7500.00,0.6068
2,RDR-00003,1342.734041,110174.8831,0.224700,0.039057,0.269743,1,36724.961,7600.43,0.1989,...,0.4258,0.2894,0,20000,180,2,0.5446,0.8530,3333.33,0.1422
3,RDR-00004,1336.267375,106901.3900,0.182539,0.017895,0.231194,2,35633.800,6679.76,0.1875,...,0.4180,0.3264,0,20000,120,3,0.5613,0.8530,5000.00,0.2132
4,RDR-00005,875.495432,70915.1300,0.184480,0.044000,0.227222,2,23638.380,6983.09,0.2954,...,0.4339,0.3006,0,50000,30,4,2.1152,5.8504,50000.00,5.8504


In [5]:
# Separate features and target
X = df[FEATURE_COLS].copy()
y = df["actual_default"].astype(int).copy()



In [6]:
df.columns

Index(['rider_id', 'avg_daily_income', 'total_income_90d',
       'income_volatility_cv', 'income_trend_90d', 'rain_season_dip',
       'income_gap_max_days', 'monthly_income_estimate', 'fuel_spend_monthly',
       'fuel_to_income_ratio', 'sacco_contrib_monthly',
       'family_remittance_monthly', 'digital_loan_monthly_out',
       'estimated_ndi', 'debt_service_ratio', 'max_safe_repayment',
       'avg_mpesa_balance', 'min_mpesa_balance', 'zero_balance_days',
       'sacco_tenure_months', 'sacco_contribution_rate', 'sacco_on_time_rate',
       'sacco_avg_days_late', 'sacco_cumulative_savings',
       'sacco_guarantor_count', 'sacco_repayment_status', 'is_sacco_member',
       'total_loans_taken', 'loans_repaid_clean', 'on_time_repayment_rate',
       'avg_days_early_late', 'max_loan_repaid', 'active_digital_loans',
       'digital_loan_frequency', 'ever_defaulted', 'restructured_loan_flag',
       'total_outstanding_debt', 'first_time_borrower', 'age',
       'months_operating', 'bik

In [ ]:
X.shape

In [ ]:
X.describe

In [ ]:
print("DATA LOADED")
print()
print(f"  X shape:      {X.shape}")
print(f"  y shape:      {y.shape}")
print(f"  Features:     {len(FEATURE_COLS)}")
print(f"  Default rate: {y.mean():.1%}")
print()
print(f"  Non-defaulters: {(y==0).sum()}")
print(f"  Defaulters:     {(y==1).sum()}")

In [7]:
# Add noise to break the perfect synthetic data signal
# This gives us a realistic AUC instead of a fake 0.99

np.random.seed(42)

noise_cols = [
    "sacco_contribution_rate",
    "sacco_on_time_rate",
    "sacco_tenure_months",
    "on_time_repayment_rate",
    "ever_defaulted",
    "estimated_ndi",
    "active_digital_loans",
]

for col in noise_cols:
    if col in X.columns:
        std = X[col].std()
        if std > 0:
            noise = np.random.normal(0, std * 0.25, size=len(X))
            X[col] = (X[col] + noise).clip(
                X[col].quantile(0.01),
                X[col].quantile(0.99)
            )

print("✅ Noise added")
print("   Level: 25% of each feature standard deviation")
print("   These features now have realistic uncertainty")

✅ Noise added
   Level: 25% of each feature standard deviation
   These features now have realistic uncertainty


In [13]:
# Step 1: Split off test set first and lock it away
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Step 2: Split remaining into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.176,
    random_state=42,
    stratify=y_temp
)



In [14]:
print()
print(f"  Training set:   {len(X_train)} riders  "
      f"({y_train.mean():.1%} default rate)")
print(f"  Validation set: {len(X_val)} riders  "
      f"({y_val.mean():.1%} default rate)")
print(f"  Test set:       {len(X_test)} riders  "
      f"({y_test.mean():.1%} default rate)")
print()



  Training set:   700 riders  (29.7% default rate)
  Validation set: 150 riders  (29.3% default rate)
  Test set:       150 riders  (29.3% default rate)



In [ ]:
print("BEFORE SMOTE:")
print(f"  Non-defaulters: {(y_train==0).sum()}")
print(f"  Defaulters:     {(y_train==1).sum()}")
print()

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("AFTER SMOTE:")
print(f"  Non-defaulters: {(y_train_bal==0).sum()}")
print(f"  Defaulters:     {(y_train_bal==1).sum()}")
print()


In [13]:
# Class weight ratio for XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")
print()

xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    early_stopping_rounds=20,   # ✅ HERE, not in fit()
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

scale_pos_weight: 1.82

[0]	validation_0-auc:0.72962
[50]	validation_0-auc:0.82669
[73]	validation_0-auc:0.81949


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=20,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=-1, num_parallel_tree=None, ...)

In [20]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    max_depth=5,
    num_leaves=31,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[
        early_stopping(stopping_rounds=20),
        log_evaluation(50)   # prints every 50 rounds
    ]
)

Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[2]	valid_0's auc: 0.822992	valid_0's binary_logloss: 0.66266


LGBMClassifier(class_weight='balanced', colsample_bytree=0.8,
               learning_rate=0.05, max_depth=5, n_estimators=1000, n_jobs=-1,
               random_state=42, subsample=0.8)

In [21]:
print(lgb_model.best_iteration_)
print(lgb_model.best_score_)

2
defaultdict(<class 'collections.OrderedDict'>, {'valid_0': OrderedDict({'auc': np.float64(0.8229916358685081), 'binary_logloss': np.float64(0.6626600079012391)})})


In [12]:
import xgboost as xgb
print(xgb.__version__)
print(xgb.__file__)

3.2.0
C:\Users\hp\anaconda3\Lib\site-packages\xgboost\__init__.py


In [ ]:
print("Before drop:", X.columns)

X = X.drop(columns=["rider_id"], errors="ignore")

print("After drop:", X.columns)

In [31]:
THRESHOLD = 0.35

final_test_pred = (ensemble_val_proba >= THRESHOLD).astype(int)

In [32]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, final_test_pred))
print(confusion_matrix(y_test, final_test_pred))

              precision    recall  f1-score   support

           0       0.62      0.19      0.29        97
           1       0.35      0.79      0.48        53

    accuracy                           0.40       150
   macro avg       0.48      0.49      0.38       150
weighted avg       0.52      0.40      0.36       150

[[18 79]
 [11 42]]


In [34]:
xgb_test_proba = xgb_model.predict_proba(X_test)[:, 1]
lgb_test_proba = lgb_model.predict_proba(X_test)[:, 1]

In [38]:
ensemble_test_proba = (xgb_test_proba * 0.55 +
                       lgb_test_proba * 0.45)

In [43]:
THRESHOLD = 0.40

final_test_pred = (ensemble_test_proba >= THRESHOLD).astype(int)

In [44]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, final_test_pred))
print(confusion_matrix(y_test, final_test_pred))

              precision    recall  f1-score   support

           0       0.92      0.47      0.63        97
           1       0.49      0.92      0.64        53

    accuracy                           0.63       150
   macro avg       0.71      0.70      0.63       150
weighted avg       0.77      0.63      0.63       150

[[46 51]
 [ 4 49]]


In [8]:
# Instead of feature noise — flipping some labels randomly
# This directly breaks the perfect feature-to-label relationship
# 15% of labels get randomly flipped
# This simulates real world unpredictability

np.random.seed(42)

# Start fresh — reload X and y without any feature noise
df = pd.read_csv("features.csv")
with open("feature_cols.json", "r") as f:
    FEATURE_COLS = json.load(f)

X = df[FEATURE_COLS].copy()
y = df["actual_default"].astype(int).copy()

print(f"Before label noise:")
print(f"  Defaulters:     {y.sum()}")
print(f"  Non-defaulters: {(y==0).sum()}")
print(f"  Default rate:   {y.mean():.1%}")
print()

# Flip 15% of labels randomly
# This means:
#   Some riders who should default are labelled as not defaulting
#   Some riders who should not default are labelled as defaulting
#   The model cannot perfectly memorise the pattern anymore
flip_mask = np.random.random(size=len(y)) < 0.15
y_noisy   = y.copy()
y_noisy[flip_mask] = 1 - y_noisy[flip_mask]

print(f"After label noise (15% flipped):")
print(f"  Labels flipped: {flip_mask.sum()}")
print(f"  Defaulters:     {y_noisy.sum()}")
print(f"  Non-defaulters: {(y_noisy==0).sum()}")
print(f"  Default rate:   {y_noisy.mean():.1%}")
print()
print("✅ Label noise added")
print("   Model can no longer perfectly memorise the pattern")

Before label noise:
  Defaulters:     296
  Non-defaulters: 704
  Default rate:   29.6%

After label noise (15% flipped):
  Labels flipped: 166
  Defaulters:     354
  Non-defaulters: 646
  Default rate:   35.4%

✅ Label noise added
   Model can no longer perfectly memorise the pattern


In [9]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_noisy,          # ← y_noisy instead of y
    test_size=0.15,
    random_state=42,
    stratify=y_noisy     # ← y_noisy instead of y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.176,
    random_state=42,
    stratify=y_temp
)

print(f"Training set:   {len(X_train)} riders ({y_train.mean():.1%} default rate)")
print(f"Validation set: {len(X_val)} riders ({y_val.mean():.1%} default rate)")
print(f"Test set:       {len(X_test)} riders ({y_test.mean():.1%} default rate)")

Training set:   700 riders (35.4% default rate)
Validation set: 150 riders (35.3% default rate)
Test set:       150 riders (35.3% default rate)


In [19]:
print("FINAL TEST SET EVALUATION")
print(f"Running on {len(X_test)} riders model has never seen")
print()

test_metrics = evaluate_model(
    y_test,
    ensemble_test_proba,
    ensemble_test_pred,
    "ENSEMBLE — FINAL TEST SET"
)

FINAL TEST SET EVALUATION
Running on 150 riders model has never seen



NameError: name 'evaluate_model' is not defined

In [24]:
import xgboost as xgb
print(xgb.__version__)

3.2.0


In [ ]:
from sklearn.metrics import roc_auc_score

y_pred = lgb_model.predict_proba(X_val)[:,1]
print(roc_auc_score(y_val, y_pred))

In [22]:
from sklearn.metrics import recall_score
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.05)

for t in thresholds:
    preds = (ensemble_val_proba >= t).astype(int)
    recall = recall_score(y_val, preds)
    print(f"Threshold: {t:.2f}, Recall: {recall:.4f}")

NameError: name 'ensemble_val_proba' is not defined

In [23]:
# XGBoost
xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]

# LightGBM
lgb_val_proba = lgb_model.predict_proba(X_val)[:, 1]

# Ensemble
ensemble_val_proba = (xgb_val_proba * 0.55 +
                      lgb_val_proba * 0.45)

In [26]:
## finding the relationship between Recall and Precision
from sklearn.metrics import recall_score
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.05)

for t in thresholds:
    preds = (ensemble_val_proba >= t).astype(int)
    recall = recall_score(y_val, preds)
    print(f"Threshold: {t:.2f}, Recall: {recall:.4f}")

Threshold: 0.10, Recall: 1.0000
Threshold: 0.15, Recall: 1.0000
Threshold: 0.20, Recall: 1.0000
Threshold: 0.25, Recall: 1.0000
Threshold: 0.30, Recall: 1.0000
Threshold: 0.35, Recall: 0.9811
Threshold: 0.40, Recall: 0.9057
Threshold: 0.45, Recall: 0.7358
Threshold: 0.50, Recall: 0.6038
Threshold: 0.55, Recall: 0.5849
Threshold: 0.60, Recall: 0.3962
Threshold: 0.65, Recall: 0.2264
Threshold: 0.70, Recall: 0.1132
Threshold: 0.75, Recall: 0.0000
Threshold: 0.80, Recall: 0.0000
Threshold: 0.85, Recall: 0.0000


In [25]:
## precision and Recall
from sklearn.metrics import precision_score

for t in [0.25, 0.30, 0.35, 0.40]:
    preds = (ensemble_val_proba >= t).astype(int)
    recall = recall_score(y_val, preds)
    precision = precision_score(y_val, preds)
    print(f"T: {t}, Recall: {recall:.3f}, Precision: {precision:.3f}")

T: 0.25, Recall: 1.000, Precision: 0.353
T: 0.3, Recall: 1.000, Precision: 0.368
T: 0.35, Recall: 0.981, Precision: 0.430
T: 0.4, Recall: 0.906, Precision: 0.545


In [ ]:
test_metrics = evaluate_model(
    y_test,
    ensemble_test_proba,
    ensemble_test_pred,
    "ENSEMBLE — FINAL TEST SET"
)

In [45]:
# Try different thresholds and see the trade-off
print("THRESHOLD ANALYSIS")
print()
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<10} {'F1':<10} {'Defaulters Caught'}")
print("-" * 60)

from sklearn.metrics import precision_score, recall_score, f1_score

for thresh in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    preds     = (ensemble_test_proba >= thresh).astype(int)
    precision = precision_score(y_test, preds, zero_division=0)
    recall    = recall_score(y_test, preds, zero_division=0)
    f1        = f1_score(y_test, preds, zero_division=0)
    caught    = preds[y_test == 1].sum()

    print(f"  {thresh:<10}   {precision:<10.3f}   {recall:<8.3f}   "
          f"{f1:<8.3f}   {caught}/{y_test.sum()} defaulters")

THRESHOLD ANALYSIS

Threshold    Precision    Recall     F1         Defaulters Caught
------------------------------------------------------------
  0.25         0.353        1.000      0.522      53/53 defaulters
  0.3          0.356        0.981      0.523      52/53 defaulters
  0.35         0.413        0.981      0.581      52/53 defaulters
  0.4          0.490        0.925      0.641      49/53 defaulters
  0.45         0.528        0.717      0.608      38/53 defaulters
  0.5          0.618        0.642      0.630      34/53 defaulters


In [50]:

os.makedirs("models", exist_ok=True)

# Save models
joblib.dump(xgb_model, "models/xgb_model.pkl")
joblib.dump(lgb_model, "models/lgb_model.pkl")

# ---- REAL METRICS (NO HARDCODING) ----
auc = roc_auc_score(y_test, ensemble_test_proba)
recall = recall_score(y_test, final_test_pred)
precision = precision_score(y_test, final_test_pred)
f1 = f1_score(y_test, final_test_pred)

model_config = {
    "threshold": THRESHOLD,
    "xgb_weight": XGB_WEIGHT,
    "lgb_weight": LGB_WEIGHT,
    "feature_cols": FEATURE_COLS,
    "n_features": len(FEATURE_COLS),
    "performance": {
        "auc": float(auc),
        "recall": float(recall),
        "precision": float(precision),
        "f1": float(f1),
        "threshold": THRESHOLD
    },
    "note": "Trained on synthetic data. Retrain on real SACCO data before production."
}

with open("models/model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)

print("✅ All models saved")
print()
print("   models/xgb_model.pkl")
print("   models/lgb_model.pkl")
print("   models/model_config.json")
print()
print("MODEL SUMMARY")
print(f"  AUC-ROC:    {auc:.3f}")
print(f"  Recall:     {recall:.3f}")
print(f"  Precision:  {precision:.3f}")
print(f"  F1 Score:   {f1:.3f}")
print(f"  Threshold:  {THRESHOLD}")
print()
print("=" * 45)
print("  MODEL TRAINING COMPLETE")
print("  Next: Code Phase 4 — Scoring Pipeline")
print("=" * 45)

✅ All models saved

   models/xgb_model.pkl
   models/lgb_model.pkl
   models/model_config.json

MODEL SUMMARY
  AUC-ROC:    0.789
  Recall:     0.925
  Precision:  0.490
  F1 Score:   0.641
  Threshold:  0.4

  MODEL TRAINING COMPLETE
  Next: Code Phase 4 — Scoring Pipeline


In [47]:
XGB_WEIGHT = 0.55
LGB_WEIGHT = 0.45